# SoccerNet GameState Challenge

## Challenge description

GameState Challenge by SoccerNet provides a baseline as a help to start working on the proposed task. The task itself consists in the development of a computer vision model able to performantly label objects (players, goalkeepers, referees and other), pitch lines and camera from a 30-second soccer game video.

The baseline contains *json* labelling files from example videos as an input for training, testing and validating the model (as a groundtruth), and has 37 more videos to evaluate it. These videos are splitted into 750 *jpg* images (30 seconds at 25 fps) each of them containing labeling data in the *json* file related to that video, in order to know the objects position (w.r.t. the camera viewpoint and pitch), and pitch lines (w.r.t. camera viewpoint).

### Guidelines

- The prize is USD 1000.
- The evaluation metric is the GS-HOTA, a modified version of the HOTA metric.
- The deadline to submit the results is May 30th 2024 at 11.59 pm Pacific Time.
- To be eligible for the prize, the team must provide a short report describing the details of the methodology (CVPR format, max 2 pages), with a short demo video on one SoccerNet challenge sequence showcasing their solution.
- The evlauation server is open until May 30th and the deadline for submitting the report is June 6th.

### Game State Reconstruction

**Maintain a complete understanding of the game by predicting:**
- top-view player localization (optionally with camera calibration)
- role classification (player, goalkeeper, referee, other)
- team affiliation and team side (left, right)
- jersey number recognition

### Observations

- Knowing a player's, goalkeeper's or referee's position w.r.t. the camera viewpoint provides the opportunity to render rectangles around them (for example, containing their jersey number and team information if applicable); and by knowing pitch lines information, they can be higlighted. 
- On the other hand, by knowing a player's, goalkeeper's and referee's position w.r.t. to the pitch, it is possible to render them as dots in a pitch minimap.

## Data Structure

![title](assets/tree.png)

- **sequences_info.json** is duplicated in the 4 directories. It provides to which directory each video belongs (train, valid, test, challenge). See code in next cell.
- **SNGS-... folders** ("..." represents a 3 digit number). There is one of these folders per video. *sequences_info.json* details if that video is for training, validation, testing or for the challenge itself.
- **Labels-GameState.json** describes every observation made by computer vision in each frame. Further develop in next section.
- __*.jpg files__ are located in each *img1* directory, which contains 750 of them (750 frames of a 30 seconds video at 25 fps).

#### Print the number of folders (videos) and files in data directory (and not its children directories) and check if sequences_info.json files in different directories are the same

In [5]:
import os
from data_analysis import count_folders_and_files, check_duplicated_sequence_files, DIRECTORIES_PATH, DATA_DIRECTORIES

for directory in DATA_DIRECTORIES:
    path = os.path.join(DIRECTORIES_PATH, directory)
    folder_count, file_count, file_name = count_folders_and_files(path)
    print(f"Directory: {directory}")
    print(f"Number of folders (videos): {folder_count}")
    print(f"Number of files: {file_count}")
    print(f"Names of those files: {file_name} \n")

print("Sequences files are the same.\n") if check_duplicated_sequence_files() else print("Sequences files are different.\n")

Directory: challenge
Number of folders (videos): 37
Number of files: 1
Names of those files: ['sequences_info.json'] 

Directory: test
Number of folders (videos): 50
Number of files: 1
Names of those files: ['sequences_info.json'] 

Directory: train
Number of folders (videos): 58
Number of files: 1
Names of those files: ['sequences_info.json'] 

Directory: valid
Number of folders (videos): 59
Number of files: 1
Names of those files: ['sequences_info.json'] 

Sequences files are the same.



### Labels-GameState.json
As briefly described above, the structure of the file is devided into 1 object (**info**) and 3 lists of objects (**images**, **annotations**, **categories**).
This file is contained in the **train**, **test** and **valid** directories but not in **challenge** as this is the ground truth of the model (the one to be compared with when evaluated).

#### { } info

> Information about the game and video.
> Except for **frame_rate** and **seq_lenght** (both number), the properties are strings.

- **version**: The version of the processing software used. At March 2024, the requested version is 1.1.
- **game_id**: The identifier of the game.
- **id**: The identifier of the clip.
- **num_tracklets**: The number of tracklets in the clip.
- **action_position**: N/A
- **action_class**: The class of the action (e.g., "Corner").
- **visibility**: The visibility of the action (e.g., "visible").
- **game_time_start**: The start time of the clip in game time.
- **game_time_stop**: The end time of the clip in game time.
- **clip_start**: The start time of the clip in milliseconds.
- **clip_stop**: The end time of the clip in milliseconds.
- **name**: The name of the clip.
- **im_dir**: The directory of the images. Always "img1"
- **frame_rate**: The frame rate of the video. Always 25.
- **eq_length**: The length of the sequence in frames. Always 750.
- **im_ext**: The extension of the image files. Always ".jpg"

**Example**

![title](assets/info_example_json.png)

#### [ ] categories

> There are 7 categories used to relate recognitions in the annotations (see anotations section below).

| **supercategory** | **category** | **id** |
|-------------------|--------------|--------|
| object            | player       | 1      |
| object            | goalkeeper   | 2      |
| object            | referee      | 3      |
| object            | ball         | 4      |
| pitch             | pitch        | 5      |
| camera            | camera       | 6      |
| object            | other        | 7      |

- Every category but **pitch** contains the same properties, while pitch contains all of those, plus **lines** (list of strings).

**Example of object supercategory structure**


![title](assets/categories_sup_object_example_json.png)

**Example of pitch supercategory structure**

![title](assets/categories_sup_pitch_example_json.png)

#### [ ] images

> Array of objects, each representing a frame.

- **is_labeled**: A boolean indicating whether the frame is labeled.
- **image_id**: The identifier of the image.
- **file_name**: The name of the image file.
- **height**: The height of the image in pixels.
- **width**: The width of the image in pixels.
- **has_labeled_person**: A boolean indicating whether the frame has a labeled person.
- **has_labeled_pitch**: A boolean indicating whether the frame has a labeled pitch.
- **has_labeled_camera**: A boolean indicating whether the frame has a labeled camera.
- **ignore_regions_y**: An array of y-coordinates to ignore.
- **ignore_regions_x**: An array of x-coordinates to ignore.

**Example**

![title](assets/images_example_json.png)

#### [ ] annotations

> Contains detailed information about the labeled entities in each frame (each object corresponding to a labeled entity).

**Properties of every supercategory are**
- **id**: A unique identifier for the annotation.
- **image_id**: The image this annotation is associated with.
- **supercategory**: Identifies the supercategory to which the category belongs.
- **category_id**: Identifier of the category of the labeled entity (e.g., person, pitch, camera).

##### **"supercategory": "object"**

- **track_id** N/A
- **attributes**
    - **role** specifies the category of the recognized object (see categories of supercategory object).
    - **jersey** always null but when role equals player.
    - **team** not null when role equals player or goalkeeper. Values are **right** or **left** w.r.t. camera viewpoint.
- **bbox_image** contains a dictionary with the object coordinates in the image. **x** and **y** represent the starting point of the box (top left corner).
- **bbox_pitch** contains a dictionary with the object coordinates in the pitch. 1 unit in the data represents 1 meter in the pitch. Central circle is the coordinates origin. i.e.: When object is on the right side of the pitch (w.r.t. camera viewpoint), its **x**-coordinates' values must be greater than zero.


In [ ]:
from data_analysis import check_goalkeeper_jersey
check_goalkeeper_jersey()

**Example of annotation**

![title](assets/annotation_object_example_json.png)

**Example of rendered image**

![title](assets/object_rendering_example.png)

##### **"supercategory": "pitch"**

- **lines** (see categories section), is a list of dictionaries containing 2D coordinates (x,y) of points from a pitch line.
    - Each coordinate must be multiplied times the image resolution in order to be mapped in the image.
    - Lines are not required (they depend on the identification in the frame).

**Example of annotation**

In [1]:
from data_analysis import show_random_pitch_annotation
show_random_pitch_annotation()

{
    "id": "3116000021",
    "image_id": "3116000001",
    "supercategory": "pitch",
    "category_id": 5,
    "lines": {
        "Side line top": [
            {
                "x": 0.0,
                "y": 0.385512962962963
            },
            {
                "x": 0.12891666666666668,
                "y": 0.37255000000000005
            },
            {
                "x": 0.15859335937500002,
                "y": 0.3692706018518519
            },
            {
                "x": 0.18827005208333333,
                "y": 0.3659912037037038
            },
            {
                "x": 0.24762343750000002,
                "y": 0.3594324074074074
            }
        ],
        "Side line right": [
            {
                "x": 0.25053125,
                "y": 0.3595870370370371
            },
            {
                "x": 0.5435,
                "y": 0.4341240740740741
            },
            {
                "x": 0.83646875,
                "y": 0.50

**Example of rendered image**

![title](assets/pitch_lines_rendering_example.png)

##### **"supercategory": "camera"**

- **parameters** contains camera calibration metadata.

**Examples of camera annotations with different (and not null) parameters**
> No patterns were found between the two diferent layouts (displayed below) of camera annotations where its image has **"has_labeled_camera": true**.

![title](assets/annotation_camera_example1_json.png)

![title](assets/annotation_camera_example2_json.png)

> List of frames of video SNGS-016 whose **has_labeled_camera** property is false
> Observation: **parameters** is null in every case.

In [1]:
from data_analysis import print_camera_annotations
print_camera_annotations()

{
    "id": "3116007220",
    "image_id": "3116000348",
    "supercategory": "camera",
    "category_id": 6,
    "parameters": null,
    "relative_mean_reproj": null,
    "accuracy@5": null
}
{
    "id": "3116007854",
    "image_id": "3116000382",
    "supercategory": "camera",
    "category_id": 6,
    "parameters": null,
    "relative_mean_reproj": null,
    "accuracy@5": null
}
{
    "id": "3116008101",
    "image_id": "3116000397",
    "supercategory": "camera",
    "category_id": 6,
    "parameters": null,
    "relative_mean_reproj": null,
    "accuracy@5": null
}
{
    "id": "3116008503",
    "image_id": "3116000424",
    "supercategory": "camera",
    "category_id": 6,
    "parameters": null,
    "relative_mean_reproj": null,
    "accuracy@5": null
}
{
    "id": "3116008545",
    "image_id": "3116000427",
    "supercategory": "camera",
    "category_id": 6,
    "parameters": null,
    "relative_mean_reproj": null,
    "accuracy@5": null
}
{
    "id": "3116008689",
    "image_id"